# Example 00b: subdividing passive polygons

This notebook shows how to make a passive machine description finer by subdividing quadrilateral passive polygons into smaller child polygons. It deliberately uses a mixed passive description containing both quadrilateral and non-quadrilateral polygons.

The distinction matters because topology subdivision uses a bilinear quadrilateral map. Four-vertex polygons can be split into smaller four-vertex child polygons. General polygons are still valid passive structures for the usual `G` and `LH` passive refinement modes, but they are not subdivided by this helper and are not compatible with the `GQ` quadrature mode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

from freegsnke import build_machine
from freegsnke.refine_passive import polygon_area, subdivide_passive_polygons

Define a small mixed passive description. The first two entries are quadrilaterals and can be subdivided. The last entry has five vertices; it is a valid polygonal passive for the standard grid refinement path, but it cannot be split by the quadrilateral subdivision helper.

In [ ]:
resistivity_wall = 5.5e-7

passive_coils = [
    {
        "R": [0.30, 1.90, 1.90, 0.30],
        "Z": [-1.35, -1.35, -1.45, -1.45],
        "name": "lower_quad_wall",
        "resistivity": resistivity_wall,
        "current_multiplier": 0.45,
    },
    {
        "R": [0.35, 0.45, 0.45, 0.35],
        "Z": [-1.20, -1.20, 1.20, 1.20],
        "name": "left_quad_wall",
        "resistivity": resistivity_wall,
        "current_multiplier": 0.35,
    },
    {
        "R": [0.85, 1.10, 1.30, 1.18, 0.92],
        "Z": [-0.18, -0.28, 0.00, 0.26, 0.22],
        "name": "nonquad_baffle",
        "resistivity": resistivity_wall,
        "current_multiplier": 0.20,
    },
]

In [ ]:
def vertex_count(passive):
    return int(np.size(passive["R"]))

for passive in passive_coils:
    print(
        f"{passive['name']}: {vertex_count(passive)} vertices, "
        f"area={polygon_area(passive['R'], passive['Z']):.4f} m^2"
    )

The default behaviour is strict. Calling `subdivide_passive_polygons` on a mixed list raises an error as soon as it finds the non-quadrilateral polygon. This is useful when the intention is to subdivide every polygon and any unchanged polygon would be a modelling mistake.

In [ ]:
try:
    subdivide_passive_polygons(passive_coils, max_edge_length=0.35)
except ValueError as err:
    print(err)

For a mixed machine, set `non_quadrilateral="keep"`. Quadrilateral parents are replaced by children; non-quadrilateral passives are copied through unchanged. The child entries keep `parent_name`, `parent_index`, `subdivision_index`, `subdivision_shape`, `parent_area`, and `area_fraction`, so reconstructed currents assigned to a parent can be redistributed to its children.

In [ ]:
refined_passive_coils = subdivide_passive_polygons(
    passive_coils,
    max_edge_length=0.35,
    non_quadrilateral="keep",
)

print(f"Original passive count: {len(passive_coils)}")
print(f"Refined passive count: {len(refined_passive_coils)}")
for passive in refined_passive_coils:
    parent = passive.get("parent_name", passive["name"])
    print(
        f"{passive['name']:<22} vertices={vertex_count(passive)} "
        f"parent={parent:<16} multiplier={passive['current_multiplier']:.6f}"
    )

For each quadrilateral parent, the child `current_multiplier` values sum to the original parent value. The unchanged non-quadrilateral keeps its original multiplier. This is the mechanism used to preserve reconstructed parent currents after subdivision.

In [ ]:
for parent in passive_coils:
    child_multiplier_sum = sum(
        child["current_multiplier"]
        for child in refined_passive_coils
        if child.get("parent_name", child["name"]) == parent["name"]
    )
    print(
        f"{parent['name']}: original={parent['current_multiplier']:.6f}, "
        f"after refinement={child_multiplier_sum:.6f}"
    )

In [ ]:
def plot_passives(axis, passives, title):
    axis.set_title(title)
    axis.set_aspect("equal")
    axis.set_xlim(0.15, 2.05)
    axis.set_ylim(-1.60, 1.35)
    for passive in passives:
        vertices = np.column_stack([passive["R"], passive["Z"]])
        is_child = "parent_name" in passive
        patch = Polygon(
            vertices,
            closed=True,
            facecolor="tab:blue" if is_child else "lightgrey",
            edgecolor="black",
            linewidth=0.8,
            alpha=0.45 if is_child else 0.70,
        )
        axis.add_patch(patch)
        centroid = vertices.mean(axis=0)
        axis.text(centroid[0], centroid[1], passive["name"], ha="center", va="center", fontsize=7)
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")

fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=120)
plot_passives(axes[0], passive_coils, "Original mixed passives")
plot_passives(axes[1], refined_passive_coils, "After quad subdivision")
plt.tight_layout()

The refined list can be passed directly to `build_machine.tokamak`. Because the machine still contains the non-quadrilateral polygon, use the standard `G` or `LH` refinement mode. If `refine_mode="GQ"` were used here, the build would fail on the five-vertex polygon because `GQ` currently requires four vertices.

In [ ]:
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/example/active_coils.pickle",
    passive_coils_data=refined_passive_coils,
    limiter_path="../machine_configs/example/limiter.pickle",
    wall_path="../machine_configs/example/wall.pickle",
    magnetic_probe_path="../machine_configs/example/magnetic_probes.pickle",
    refine_mode="G",
)

print(f"Tokamak passive structures: {tokamak.n_passive_coils}")

In [ ]:
try:
    build_machine.tokamak(
        active_coils_path="../machine_configs/example/active_coils.pickle",
        passive_coils_data=refined_passive_coils,
        limiter_path="../machine_configs/example/limiter.pickle",
        wall_path="../machine_configs/example/wall.pickle",
        magnetic_probe_path="../machine_configs/example/magnetic_probes.pickle",
        refine_mode="GQ",
    )
except ValueError as err:
    print("GQ build with a non-quadrilateral passive:")
    print(err)